# 03 · Análisis de Progenitores del ICL y BCG
**Metodología:** §7 de Mayes+2026

1. **§7.1** – Masa media de progenitores ICL vs BCG (Fig. 13)
2. **§7.2** – Fracción de progenitores compartidos ICL↔BCG (Fig. 14)
3. **§7.3** – Localización de material de progenitores alta/baja masa (Fig. 15)
4. **§7.4** – Progenitor más significativo por componente


In [ ]:
import sys, os, pickle
import numpy as np
import h5py
import matplotlib.pyplot as plt
from astropy.cosmology import FlatLambdaCDM
from scipy.stats import linregress
from scipy.interpolate import interp1d
import warnings
warnings.filterwarnings('ignore')

sys.path.insert(0, './original_shift_code')
import illustris_python as il
import Catalogue
import params_icl as P

%matplotlib inline
plt.rcParams.update({'figure.dpi': 110, 'font.size': 12,
                     'axes.spines.top': False, 'axes.spines.right': False})

cosmo = FlatLambdaCDM(H0=67.74, Om0=0.3089)
Header = il.groupcat.loadHeader(P.basePath, P.SNAP)

with h5py.File(P.CATALOG_OUT, 'r') as f:
    group_idx   = f['group_idx'][:]
    M200c       = f['M200c_Msun'][:]
    GroupPos    = f['GroupPos_kpc'][:]
    bcg_sub_idx = f['bcg_sub_idx'][:]
    dyn_state   = f['dyn_state'][:] if 'dyn_state' in f else None

n_cl = len(group_idx)
lM   = np.log10(M200c)
COLORS_STATE = {0:'#2196F3', 1:'#FF9800', 2:'#F44336'}
LABELS_STATE = {0:'Relajado', 1:'Intermedio', 2:'Perturbado'}
print(f"Catálogo cargado: {n_cl} cúmulos")


In [ ]:
# Cargar catálogo de ensamblaje estelar
with h5py.File(P.SA_FILE, 'r') as f:
    sa_pid      = f['ParticleID'][:]
    sa_channel  = f['AssemblyChannel'][:]
    sa_progmass = f['ProgGalaxyMass'][:] * P.UM
    sa_progid   = f['ProgGalaxyID'][:]

sa_map = dict(zip(sa_pid, np.arange(len(sa_pid))))
print(f"Catálogo SA: {len(sa_pid):,} partículas")


In [ ]:
# Funciones auxiliares (mismas que notebook 02)
def rotate_by_inertia_tensor(pos_rel, mass, r_lim=np.inf):
    dist = np.linalg.norm(pos_rel, axis=1)
    ok   = (dist>0)&(dist<=r_lim)&np.isfinite(mass)
    p,m  = pos_rel[ok], mass[ok]
    if m.sum()==0 or len(m)<4: return pos_rel, np.eye(3)
    w = 1/dist[ok]**2; mt = m.sum()
    Ixx=np.sum(m*p[:,0]**2*w)/mt; Iyy=np.sum(m*p[:,1]**2*w)/mt
    Izz=np.sum(m*p[:,2]**2*w)/mt; Ixy=np.sum(m*p[:,0]*p[:,1]*w)/mt
    Ixz=np.sum(m*p[:,0]*p[:,2]*w)/mt; Iyz=np.sum(m*p[:,1]*p[:,2]*w)/mt
    I=np.array([[Ixx,Ixy,Ixz],[Ixy,Iyy,Iyz],[Ixz,Iyz,Izz]])
    ev,evec=np.linalg.eigh(I)
    R=evec[:,np.argsort(ev)[::-1]].T
    return pos_rel@R.T, R

def sb_and_holmberg(r_2d, lum_r, n_bins=60):
    r_bins=np.logspace(np.log10(0.5),np.log10(np.percentile(r_2d,99)),n_bins+1)
    r_mid=np.sqrt(r_bins[:-1]*r_bins[1:]); mu_r=np.full(n_bins,np.nan)
    for k,(r1,r2) in enumerate(zip(r_bins[:-1],r_bins[1:])):
        mk=(r_2d>=r1)&(r_2d<r2)
        if mk.sum()==0: continue
        sl=lum_r[mk].sum()/(np.pi*((r2*1e3)**2-(r1*1e3)**2))
        if sl>0: mu_r[k]=P.SB_CONST-2.5*np.log10(sl)
    valid=np.isfinite(mu_r)&(r_mid>0)
    r_h=np.nan
    if valid.sum()>=3:
        rv,mv=r_mid[valid],mu_r[valid]
        idx=np.argsort(rv); rv,mv=rv[idx],mv[idx]
        if mv[0]<=P.MU_HOLMBERG<=mv[-1]:
            try: r_h=float(interp1d(mv,rv,fill_value='extrapolate')(P.MU_HOLMBERG))
            except: pass
    return r_h

def hmr(r, m):
    if len(r)==0 or m.sum()==0: return np.nan
    idx=np.argsort(r); cum=np.cumsum(m[idx])
    i50=np.searchsorted(cum,cum[-1]/2)
    return r[idx[min(i50,len(r)-1)]]

def linfit(x, y):
    ok=np.isfinite(x)&np.isfinite(y)
    if ok.sum()<3: return np.nan,np.nan
    sl,ic,*_=linregress(x[ok],y[ok]); return sl,ic

def load_prog_data(sub_id, cen_pos, header=Header):
    """Carga partículas + clasificación SA + separa BCG/ICL."""
    fields=['Coordinates','Masses','ParticleIDs','GFM_StellarPhotometrics']
    st=il.snapshot.loadSubhalo(P.basePath,P.SNAP,int(sub_id),'stars',fields=fields)
    pos  =Catalogue.Distance_3D(st['Coordinates']*P.UL, cen_pos, header['BoxSize']*P.UL)
    mass =st['Masses']*P.UM
    pids =st['ParticleIDs']
    phot =st['GFM_StellarPhotometrics']
    channel=np.full(len(pids),-1,dtype=int)
    progm  =np.full(len(pids),np.nan)
    progid =np.full(len(pids),-1,dtype=np.int64)
    for j,pid in enumerate(pids):
        if pid in sa_map:
            k=sa_map[pid]; channel[j]=int(sa_channel[k])
            progm[j]=float(sa_progmass[k]); progid[j]=int(sa_progid[k])
    pos_rot,_=rotate_by_inertia_tensor(pos,mass)
    r_2d=np.sqrt(pos_rot[:,0]**2+pos_rot[:,1]**2)
    lum_r=10**((P.M_SUN_R_AB-phot[:,5])/2.5)
    r_h=sb_and_holmberg(r_2d,lum_r)
    mask_bcg = r_2d<=r_h if np.isfinite(r_h) else np.zeros(len(r_2d),bool)
    return {'mass':mass,'r_2d':r_2d,'channel':channel,
            'progmass':progm,'progid':progid,'lum_r':lum_r,
            'mask_bcg':mask_bcg,'mask_icl':~mask_bcg,'r_h':r_h}


## §7.1 – Masa media de progenitores ICL vs BCG (Fig. 13)

In [ ]:
print("§7.1 – Masa media de progenitores...")
mean_pm_icl = np.full(n_cl, np.nan)
mean_pm_bcg = np.full(n_cl, np.nan)
frac_hi_icl = np.full(n_cl, np.nan)   # fracción ICL de progenitores M>thresh

for i in range(n_cl):
    try: d = load_prog_data(bcg_sub_idx[i], GroupPos[i])
    except: continue
    for mask, arr in [(d['mask_icl'], mean_pm_icl), (d['mask_bcg'], mean_pm_bcg)]:
        pm = d['progmass'][mask]; m = d['mass'][mask]
        ex = d['channel'][mask] != 0
        ok = ex & np.isfinite(pm)
        if ok.sum()>0: arr[i] = np.average(pm[ok], weights=m[ok])
    # fracción de alta masa en ICL
    ex_icl = d['mask_icl'] & (d['channel']!=0) & np.isfinite(d['progmass'])
    if ex_icl.sum()>0:
        hi = d['mass'][ex_icl & (d['progmass']>=P.M_PROG_THRESH)].sum()
        frac_hi_icl[i] = hi / d['mass'][ex_icl].sum()
    if (i+1)%10==0: print(f"  {i+1}/{n_cl}",end="\r")
print("\nListo.")
print(f"Fracción media ICL de prog. M>1e10 M☉: {np.nanmean(frac_hi_icl):.2f} ± {np.nanstd(frac_hi_icl):.2f}")


In [ ]:
fig, ax = plt.subplots(figsize=(7,6))
valid = np.isfinite(mean_pm_icl)&np.isfinite(mean_pm_bcg)
sc = ax.scatter(np.log10(mean_pm_bcg[valid]), np.log10(mean_pm_icl[valid]),
                c=lM[valid], cmap='viridis', s=25, alpha=0.8)
plt.colorbar(sc, ax=ax, label=r'$\log M_{{200c}}$')
x = np.log10(mean_pm_bcg[valid]); y = np.log10(mean_pm_icl[valid])
sl,ic = linfit(x,y)
xx = np.linspace(x.min(),x.max(),100)
ax.plot(xx,sl*xx+ic,'orange',lw=2,label=f'β={sl:.3f}')
lim=[min(x.min(),y.min()),max(x.max(),y.max())]
ax.plot(lim,lim,'k--',lw=1.2,label='1:1')
ax.set_xlabel(r'$\log\langle M_{{prog,BCG}}\rangle$')
ax.set_ylabel(r'$\log\langle M_{{prog,ICL}}\rangle$')
ax.set_title('Masa media de progenitores (Fig. 13)'); ax.legend()
pct_below = (mean_pm_icl[valid]<mean_pm_bcg[valid]).mean()*100
ax.text(0.03,0.97,f'ICL<BCG en {pct_below:.0f}% de los cúmulos',
        transform=ax.transAxes,va='top',fontsize=10)
plt.tight_layout()
plt.savefig('fig13_masa_progenitores.pdf', bbox_inches='tight')
plt.show()


## §7.2 – Progenitores compartidos ICL ↔ BCG (Fig. 14)

In [ ]:
print("§7.2 – Progenitores compartidos...")
shared_icl = np.full(n_cl, np.nan)   # fracción ICL cuyos prog también están en BCG
shared_bcg = np.full(n_cl, np.nan)   # fracción BCG cuyos prog también están en ICL

for i in range(n_cl):
    try: d = load_prog_data(bcg_sub_idx[i], GroupPos[i])
    except: continue
    ex = d['channel'] != 0
    prog_icl = set(d['progid'][d['mask_icl']&ex&(d['progid']>=0)].tolist())
    prog_bcg = set(d['progid'][d['mask_bcg']&ex&(d['progid']>=0)].tolist())
    if len(prog_icl)>0: shared_icl[i] = len(prog_icl&prog_bcg)/len(prog_icl)
    if len(prog_bcg)>0: shared_bcg[i] = len(prog_icl&prog_bcg)/len(prog_bcg)
    if (i+1)%10==0: print(f"  {i+1}/{n_cl}",end="\r")
print("\nListo.")
print(f"Progenitores compartidos ICL→BCG: {np.nanmean(shared_icl)*100:.1f}%  (paper: 93%)")
print(f"Progenitores compartidos BCG→ICL: {np.nanmean(shared_bcg)*100:.1f}%  (paper: 91%)")


In [ ]:
fig, axes = plt.subplots(1,2,figsize=(13,5))
bins = np.linspace(0,1,25)
ax = axes[0]
ax.hist(shared_icl[np.isfinite(shared_icl)],bins=bins,color='royalblue',alpha=0.7,label='ICL→BCG')
ax.hist(shared_bcg[np.isfinite(shared_bcg)],bins=bins,color='tomato',alpha=0.7,label='BCG→ICL')
for x,col in [(shared_icl,'royalblue'),(shared_bcg,'tomato')]:
    ax.axvline(np.nanmedian(x),color=col,lw=2,ls='--')
ax.set_xlabel('Fracción de progenitores compartidos'); ax.set_ylabel('N cúmulos')
ax.set_title('Progenitores compartidos (Fig. 14 top)'); ax.legend()

ax = axes[1]
if dyn_state is not None:
    for s in [0,1,2]:
        m=dyn_state==s
        ax.scatter(shared_icl[m],shared_bcg[m],color=COLORS_STATE[s],
                   label=LABELS_STATE[s],s=20,alpha=0.8)
else:
    ax.scatter(shared_icl,shared_bcg,color='steelblue',s=20,alpha=0.7)
ax.plot([0,1],[0,1],'k--',lw=1)
ax.set_xlabel('Fracción ICL→BCG'); ax.set_ylabel('Fracción BCG→ICL')
ax.set_title('Por estado dinámico (Fig. 14 bottom)'); ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig('fig14_progenitores_compartidos.pdf', bbox_inches='tight')
plt.show()


## §7.3 – HMR del material por masa de progenitor (Fig. 15)

In [ ]:
print("§7.3 – HMR por masa de progenitor...")
# Para cada cúmulo: HMR del material de progenitores alta (>thresh) y baja masa
# separado por canal (todo, mergers, stripped)
results_hmr = {k: {'hi': np.full(n_cl,np.nan), 'lo': np.full(n_cl,np.nan)}
               for k in ['all','mer','str']}

for i in range(n_cl):
    try: d = load_prog_data(bcg_sub_idx[i], GroupPos[i])
    except: continue
    pm = d['progmass']; r = d['r_2d']; m = d['mass']; ch = d['channel']
    ex = ch != 0
    for key, filt in [('all', ex), ('mer', ex&(ch==1)), ('str', ex&(ch==2))]:
        for hi_lo, cond in [('hi', pm>=P.M_PROG_THRESH), ('lo', pm<P.M_PROG_THRESH)]:
            mk = filt & cond & np.isfinite(pm)
            if mk.sum()>0:
                results_hmr[key][hi_lo][i] = hmr(r[mk], m[mk])
    if (i+1)%10==0: print(f"  {i+1}/{n_cl}",end="\r")
print("\nListo.")


In [ ]:
fig, axes = plt.subplots(1,3,figsize=(15,5),sharey=True)
titles = ['Todo ex-situ','Mergers completados','Stripped (sobrev.)']
keys   = ['all','mer','str']

for ax,title,key in zip(axes,titles,keys):
    hi = results_hmr[key]['hi']
    lo = results_hmr[key]['lo']
    for arr,col,lbl in [(hi,'tomato',f'M>10¹⁰ M☉'),(lo,'royalblue',f'M<10¹⁰ M☉')]:
        ok = np.isfinite(arr)&(arr>0)
        ax.scatter(lM[ok],arr[ok],color=col,s=20,alpha=0.7,label=lbl)
        if ok.sum()>3:
            sl,ic=linfit(lM[ok],np.log10(arr[ok]))
            xx=np.linspace(lM.min(),lM.max(),100)
            ax.plot(xx,10**(sl*xx+ic),color=col,lw=1.8,ls='--',label=f'β={sl:.3f}')
    ax.set_xlabel(r'$\log M_{{200c}}$')
    ax.set_ylabel('HMR [kpc]' if ax==axes[0] else '')
    ax.set_yscale('log'); ax.set_title(f'Fig. 15 – {title}'); ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig('fig15_hmr_por_masa_progenitor.pdf', bbox_inches='tight')
plt.show()


## §7.4 – Progenitor más significativo

In [ ]:
print("§7.4 – Progenitor más significativo...")
frac_top1_icl = np.full(n_cl, np.nan)
frac_top1_bcg = np.full(n_cl, np.nan)
n_for_90_icl  = np.full(n_cl, np.nan)

for i in range(n_cl):
    try: d = load_prog_data(bcg_sub_idx[i], GroupPos[i])
    except: continue
    ex = d['channel'] != 0
    for mask, arr1, arr90 in [(d['mask_icl'], frac_top1_icl, n_for_90_icl),
                               (d['mask_bcg'], frac_top1_bcg, None)]:
        mk = mask & ex & (d['progid']>=0)
        pids_k = d['progid'][mk]; ms_k = d['mass'][mk]
        if len(pids_k)==0: continue
        m_tot = ms_k.sum()
        contrib = {p: ms_k[pids_k==p].sum() for p in np.unique(pids_k)}
        sorted_c = sorted(contrib.values(), reverse=True)
        arr1[i] = sorted_c[0] / m_tot
        if arr90 is not None:
            cumf = np.cumsum(sorted_c)/m_tot
            arr90[i] = np.searchsorted(cumf, 0.9) + 1
    if (i+1)%10==0: print(f"  {i+1}/{n_cl}",end="\r")
print("\nListo.")
print(f"Contribución top progenitor ICL: {np.nanmean(frac_top1_icl):.3f}±{np.nanstd(frac_top1_icl):.3f}  (paper: 0.27±0.12)")
print(f"Contribución top progenitor BCG: {np.nanmean(frac_top1_bcg):.3f}±{np.nanstd(frac_top1_bcg):.3f}")


In [ ]:
fig, axes = plt.subplots(1,2,figsize=(13,5))

ax = axes[0]
bins = np.linspace(0,1,25)
ax.hist(frac_top1_icl[np.isfinite(frac_top1_icl)],bins=bins,color='royalblue',alpha=0.7,label='ICL')
ax.hist(frac_top1_bcg[np.isfinite(frac_top1_bcg)],bins=bins,color='tomato',alpha=0.7,label='BCG')
for x,col in [(frac_top1_icl,'royalblue'),(frac_top1_bcg,'tomato')]:
    ax.axvline(np.nanmedian(x),color=col,lw=2,ls='--')
ax.set_xlabel('Fracción del progenitor más significativo')
ax.set_ylabel('N cúmulos'); ax.set_title('Progenitor más significativo'); ax.legend()

ax = axes[1]
ok = np.isfinite(n_for_90_icl)
sc = ax.scatter(lM[ok], n_for_90_icl[ok], c=lM[ok], cmap='viridis', s=25, alpha=0.8)
plt.colorbar(sc, ax=ax, label=r'$\log M_{{200c}}$')
ax.set_xlabel(r'$\log M_{{200c}}$')
ax.set_ylabel('N progenitores para el 90% del ICL')
ax.set_title('Progenitores necesarios para 90% ICL')

plt.tight_layout()
plt.savefig('fig_progenitor_significativo.pdf', bbox_inches='tight')
plt.show()


## Comparación con Mayes+2026

| Resultado | Mayes+2026 | Este análisis |
|-----------|-----------|---------------|
| % ICL de progenitores M>10¹⁰ M☉ | 65 ± 15% | *(ver arriba)* |
| Progenitores compartidos ICL→BCG | **93%** | *(ver arriba)* |
| Progenitores compartidos BCG→ICL | **91%** | *(ver arriba)* |
| Contribución top progenitor ICL | **0.27 ± 0.12** | *(ver arriba)* |
